## tl;dr

The previous `29.1%` score improvement is not robust.  It appears only when
partials below the residual-noise level are admitted (`minimum_snr_db=-15`
or `-20`).  With the reliable modes (`SNR >= 0 dB`), the restored Stulov
baseline scores better.

The larger mismatch with Friture is caused by the current `StringModel`, not
by least-squares leakage.  Direct model output contains essentially only odd
harmonics: total even/odd peak power is about `-105 dB`.  The benchmark
projects force into a different, ideal fixed-fixed string that contains every
mode, so its bars cannot predict the live bBworks spectrum.


## Context & Methods

This is a diagnostic and data-quality audit, not a new hammer experiment.
Production `HammerModel`, `StringModel`, waveguide, delay, loss, and benchmark
code are read-only inputs.

### Key Assumptions

- C4 fundamental: 261.626 Hz.
- Direct-output probe reads the existing `StringModel::getSamples()` at
  44.1 kHz after the restored MIDI-80 strike.
- Target estimator reconciliation uses the same Pianoteq C4 MIDI-80 WAV.
- Positive `phase1_minus_restored_db` means phase 1 is worse.


## Data

Load the four reviewed audit tables.

In [1]:
from pathlib import Path
import csv
import math

ROOT = Path.cwd()

def load_csv(name):
    with (ROOT / name).open(encoding="utf-8") as stream:
        return list(csv.DictReader(stream))

direct_modes = load_csv("direct_output_modes.csv")
target_estimators = load_csv("target_estimator_reconciliation.csv")
snr_sensitivity = load_csv("snr_threshold_sensitivity.csv")
window_sensitivity = load_csv("window_sensitivity.csv")

print(
    len(direct_modes),
    len(target_estimators),
    len(snr_sensitivity),
    len(window_sensitivity),
)


13 13 6 4


## Results

### Direct StringModel output contains only odd modes

In [2]:
odd_power = sum(
    10.0 ** (float(row["relative_peak_power_db"]) / 10.0)
    for row in direct_modes
    if row["parity"] == "odd"
)
even_power = sum(
    10.0 ** (float(row["relative_peak_power_db"]) / 10.0)
    for row in direct_modes
    if row["parity"] == "even"
)
even_over_odd_db = 10.0 * math.log10(even_power / odd_power)
print(f"even/odd peak power = {even_over_odd_db:.3f} dB")
print("strong modes:", [
    int(row["mode"])
    for row in direct_modes
    if float(row["relative_peak_power_db"]) > -60.0
])


even/odd peak power = -105.361 dB
strong modes: [1, 3, 5, 7, 9, 11, 13]


### Least-squares and FFT extraction agree on the target WAV

In [3]:
maximum_estimator_difference = max(
    abs(float(row["ls_minus_fft100_db"]))
    for row in target_estimators
)
print(
    "maximum |LS100 - FFT100| = "
    f"{maximum_estimator_difference:.3f} dB"
)
print(
    "negative-SNR modes:",
    [
        int(row["mode"])
        for row in target_estimators
        if float(row["target_snr_db"]) < 0.0
    ],
)


maximum |LS100 - FFT100| = 0.010 dB
negative-SNR modes: [8, 9, 10, 11, 12, 13]


### The model ranking reverses when noisy modes are excluded

In [4]:
for row in snr_sensitivity:
    print(
        f"SNR >= {float(row['minimum_snr_db']):5.1f} dB: "
        f"restored={float(row['restored_stulov_rmse_db']):.4f}, "
        f"phase1={float(row['chabassier_phase1_rmse_db']):.4f}, "
        f"delta={float(row['phase1_minus_restored_db']):+.4f}"
    )


SNR >=   5.0 dB: restored=0.5140, phase1=0.7831, delta=+0.2691
SNR >=   0.0 dB: restored=0.6500, phase1=1.0843, delta=+0.4343
SNR >=  -5.0 dB: restored=1.5274, phase1=1.6134, delta=+0.0860
SNR >= -10.0 dB: restored=1.6328, phase1=1.5710, delta=-0.0618
SNR >= -15.0 dB: restored=2.8093, phase1=1.8515, delta=-0.9578
SNR >= -20.0 dB: restored=3.0953, phase1=2.1947, delta=-0.9006


### The apparent improvement disappears in a 500 ms window

In [5]:
for row in window_sensitivity:
    print(
        f"{int(row['window_ms']):3d} ms: "
        f"delta={float(row['phase1_minus_restored_db']):+.4f} dB"
    )


 50 ms: delta=-0.9830 dB
100 ms: delta=-0.9006 dB
200 ms: delta=-0.9039 dB
500 ms: delta=-0.0094 dB


## Takeaways

1. Do not use the previous `-20 dB SNR / 100 ms` score for model selection.
2. The target partial estimator is numerically well-conditioned and agrees
   with an FFT peak estimator; adjacent-mode mixing is not the observed cause.
3. The current waveguide boundary implementation recirculates each travelling
   rail independently with one sign inversion per one-way delay.  Its loop
   resonance condition admits `(2k+1)f0` only.
4. Until the string topology is either corrected or explicitly included in a
   new observation-domain benchmark, force-only ideal-string scores must be
   labelled counterfactual and cannot be compared to Friture.
